<a href="https://colab.research.google.com/github/Fahad-Hafeez/phishing-ml-classifier-comparison/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas

import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.svm import SVC
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('uci-ml-phishing-dataset.csv')

In [ ]:
print("Shape:", df.shape)
print(df.head())
print(df.info())
print(df.describe())

In [ ]:
print(df['Result'].value_counts())

In [ ]:
df['Result'].value_counts().plot(kind="bar", title='Class Distribution')
print('1 (Phishing) % =', (6157/11056)*100)
print('-1 (Legitimate) % =', (4898/11056)*100)

In [ ]:
class_counts = df['Result'].value_counts()

total_instances = class_counts.sum()
phishing_count = class_counts.get(1, 0)
legitimate_count = class_counts.get(-1, 0)

phishing_percent = (phishing_count / total_instances) * 100
legitimate_percent = (legitimate_count / total_instances) * 100

# Adjust figsize to full-width IEEE layout (7x5 inches)
plt.figure(figsize=(7, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')

plt.xticks(ticks=[0, 1], labels=['Phishing', 'Legitimate'])
plt.xlabel('Class')
plt.ylabel('Number of Instances')
plt.title('Class Distribution of Phishing Websites Dataset')

# Add text annotations on top of the bars
for index, value in enumerate(class_counts.values):
    plt.text(index, value + 50, str(value), ha='center', va='bottom')

plt.tight_layout()
# Save with dpi=300 and bbox_inches='tight' as requested
plt.savefig('fig1_class_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

caption = f"Figure 1: Class distribution of the UCI Phishing Websites dataset (n = {total_instances}). Phishing instances: {phishing_count} ({phishing_percent:.2f}%). Legitimate instances: {legitimate_count} ({legitimate_percent:.2f}%)."
print(caption)

In [ ]:
print(df.isnull().sum())

In [ ]:
#Count the number of distinct elements
print(df.nunique())

In [ ]:
#Extract the unique values for every column
print(df.apply(lambda x: x.unique()))

In [ ]:
X = df.drop('Result', axis=1)
y = df['Result']

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

In [ ]:
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=SEED, stratify=y)

print(f"X Train shape: {X_train.shape}")
print(f"X Test shape: {X_test.shape}")
print(f"y Train shape: {y_train.shape}")
print(f"y Test shape: {y_test.shape}")

print("\nClass distribution in y_train:")
print(y_train.value_counts())

print("\nClass distribution in y_test:")
print(y_test.value_counts())

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")
print(f"Shape of scaled X_train: {X_train_scaled.shape}")
print(f"Shape of scaled X_test: {X_test_scaled.shape}")

In [ ]:
sm = SMOTE(random_state=SEED)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train_scaled, y_train)

print(f"Shape of X_train after SMOTE: {X_train_balanced.shape}")
print(f"Shape of y_train after SMOTE: {y_train_balanced.shape}")
print("Class distribution in y_train after SMOTE:")
print(y_train_balanced.value_counts())

In [ ]:
param_grid = [
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'l1_ratio': [0.1, 0.5, 0.9] # Added l1_ratio for elasticnet
    }
]
GSCV = GridSearchCV(LogisticRegression(random_state=SEED, max_iter=1000), param_grid, cv=5, scoring='f1_macro', n_jobs=1)
GSCV.fit(X_train_scaled, y_train)
print(GSCV.best_params_, GSCV.best_score_)

In [ ]:
best_lr = GSCV.best_estimator_
print(f'Best model:', best_lr)

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}
RandomForest = GridSearchCV(RandomForestClassifier(random_state=SEED), rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
RandomForest.fit(X_train_scaled, y_train)
print(RandomForest.best_params_, RandomForest.best_score_)

In [ ]:
best_rf = RandomForest.best_estimator_
print(best_rf)

In [ ]:
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
SVM = GridSearchCV(SVC(random_state=SEED, probability=True), svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
SVM.fit(X_train_scaled, y_train)
print(SVM.best_params_, SVM.best_score_)

In [ ]:
best_svm = SVM.best_estimator_
print(best_svm)

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluates a trained model on test data and returns a dictionary of metrics.

    Args:
        model: A trained scikit-learn model with predict and predict_proba methods.
        X_test: Test features.
        y_test: True labels for the test data.

    Returns:
        A dictionary containing accuracy, precision, recall, f1-score, and auc-roc.
    """
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    auc_roc = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
        # Determine the index of the positive class (1) in the model's classes_
        if 1 in model.classes_:
            pos_class_idx = list(model.classes_).index(1)
            auc_roc = roc_auc_score(y_test, y_proba[:, pos_class_idx], average='macro')
        else:
            print("Warning: Positive class (1) not found in model.classes_. AUC-ROC not calculated.")
    else:
        print("Warning: Model does not have predict_proba method. AUC-ROC score cannot be calculated.")

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc_roc': auc_roc
    }
    return metrics

In [ ]:
lr_eval = evaluate_model(best_lr, X_test_scaled, y_test)
print(f"Logistic Regression Evaluation:", lr_eval)

In [ ]:
rf_eval = evaluate_model(best_rf, X_test_scaled, y_test)
print(f"Random Forest Evaluation:", rf_eval)

In [ ]:
svm_eval = evaluate_model(best_svm, X_test_scaled, y_test)
print(f"SVM Evaluation:", svm_eval)

In [ ]:
pred_lr = best_lr.predict(X_test_scaled)
pred_rf = best_rf.predict(X_test_scaled)
pred_svm = best_svm.predict(X_test_scaled)

results = [
    {'Model': 'Logistic Regression', **lr_eval},
    {'Model': 'Random Forest', **rf_eval},
    {'Model': 'SVM', **svm_eval}
]

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
param_grid = [
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'l1_ratio': [0.1, 0.5, 0.9] # Added l1_ratio for elasticnet
    }
]
GSCV_balanced = GridSearchCV(LogisticRegression(random_state=SEED, max_iter=1000), param_grid, cv=5, scoring='f1_macro', n_jobs=1)
GSCV_balanced.fit(X_train_balanced, y_train_balanced)
print(GSCV_balanced.best_params_, GSCV.best_score_)

In [ ]:
best_lr_bal = GSCV_balanced.best_estimator_
print(f'Best model:', best_lr_bal)

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}
RandomForest_balanced = GridSearchCV(RandomForestClassifier(random_state=SEED), rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
RandomForest_balanced.fit(X_train_balanced, y_train_balanced)
print(RandomForest.best_params_, RandomForest.best_score_)

In [ ]:
best_rf_bal = RandomForest_balanced.best_estimator_
print(f'Best model:', best_rf_bal)

In [ ]:
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
SVM_balanced = GridSearchCV(SVC(random_state=SEED, probability=True), svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
SVM_balanced.fit(X_train_balanced, y_train_balanced)
print(SVM.best_params_, SVM.best_score_)

In [ ]:
best_svm_bal = SVM_balanced.best_estimator_
print(f'Best model:', best_svm_bal)

In [ ]:
lr_eval_bal = evaluate_model(best_lr_bal, X_test_scaled, y_test)
rf_eval_bal = evaluate_model(best_rf_bal, X_test_scaled, y_test)
svm_eval_bal = evaluate_model(best_svm_bal, X_test_scaled, y_test)

results = [
    {'Model': 'Logistic Regression (Unbalanced)', **lr_eval},
    {'Model': 'Random Forest (Unbalanced)', **rf_eval},
    {'Model': 'SVM (Unbalanced)', **svm_eval},
    {'Model': 'Logistic Regression (Balanced)', **lr_eval_bal},
    {'Model': 'Random Forest (Balanced)', **rf_eval_bal},
    {'Model': 'SVM (Balanced)', **svm_eval_bal}
]

results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
from xgboost import XGBClassifier

# Remap labels: XGBoost requires 0/1 labels (not -1/1)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_xgb     = le.fit_transform(y_train)           # -1→0, 1→1
y_test_xgb      = le.transform(y_test)
y_train_bal_xgb = le.transform(y_train_balanced)

xgb_param_grid = {
    'n_estimators':    [100, 200],
    'max_depth':       [3, 6, 9],
    'learning_rate':   [0.05, 0.1, 0.2],
    'subsample':       [0.8, 1.0],
    'colsample_bytree':[0.8, 1.0]
}

XGB = GridSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False),
    xgb_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
XGB.fit(X_train_scaled, y_train_xgb)
print("XGB best params:", XGB.best_params_)
best_xgb = XGB.best_estimator_

# Balanced variant
XGB_balanced = GridSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False),
    xgb_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
XGB_balanced.fit(X_train_balanced, y_train_bal_xgb)
best_xgb_bal = XGB_balanced.best_estimator_

# Evaluate — XGBoost predicts 0/1 so we inverse_transform for consistent labels
def evaluate_xgb(model, X_test, y_test_binary, le, y_test_orig):
    """Evaluate XGBoost; handles 0/1 → -1/1 label remapping."""
    y_pred_bin = model.predict(X_test)
    y_pred     = le.inverse_transform(y_pred_bin)   # back to -1/1
    y_proba    = model.predict_proba(X_test)[:, 1]  # prob of class 1

    return {
        'accuracy':  accuracy_score(y_test_orig, y_pred),
        'precision': precision_score(y_test_orig, y_pred, average='macro'),
        'recall':    recall_score(y_test_orig, y_pred, average='macro'),
        'f1_score':  f1_score(y_test_orig, y_pred, average='macro'),
        'auc_roc':   roc_auc_score(y_test_binary, y_proba)
    }

xgb_eval     = evaluate_xgb(best_xgb,     X_test_scaled, y_test_xgb, le, y_test)
xgb_eval_bal = evaluate_xgb(best_xgb_bal, X_test_scaled, y_test_xgb, le, y_test)

print("XGBoost (Unbalanced):", xgb_eval)
print("XGBoost (Balanced):  ", xgb_eval_bal)

# Predictions for McNemar — back in -1/1 space
pred_xgb     = le.inverse_transform(best_xgb.predict(X_test_scaled))
pred_xgb_bal = le.inverse_transform(best_xgb_bal.predict(X_test_scaled))

# --- UPDATE your results_df to include XGBoost ---
results_extended = [
    {'Model': 'Logistic Regression (Unbalanced)', **lr_eval},
    {'Model': 'Random Forest (Unbalanced)',        **rf_eval},
    {'Model': 'SVM (Unbalanced)',                  **svm_eval},
    {'Model': 'XGBoost (Unbalanced)',              **xgb_eval},
    {'Model': 'Logistic Regression (Balanced)',    **lr_eval_bal},
    {'Model': 'Random Forest (Balanced)',           **rf_eval_bal},
    {'Model': 'SVM (Balanced)',                    **svm_eval_bal},
    {'Model': 'XGBoost (Balanced)',                **xgb_eval_bal},
]
results_df_extended = pd.DataFrame(results_extended)
print(results_df_extended.to_string(index=False))

In [ ]:
import shap
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from scipy.stats import spearmanr
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
shap.initjs()

feature_names = list(X.columns)

# --- 2a. Random Forest — TreeExplainer ---
explainer_rf   = shap.TreeExplainer(best_rf)
shap_values_rf = explainer_rf.shap_values(X_test_scaled)

# Fix: Ensure sv_rf is 2D (samples, features) for class 1
if isinstance(shap_values_rf, list):
    sv_rf = shap_values_rf[1]
elif len(shap_values_rf.shape) == 3:
    sv_rf = shap_values_rf[:, :, 1]
else:
    sv_rf = shap_values_rf

# --- 2b. XGBoost — TreeExplainer ---
explainer_xgb   = shap.TreeExplainer(best_xgb)
sv_xgb = explainer_xgb.shap_values(X_test_scaled)

# --- 2c. Logistic Regression — LinearExplainer ---
background_lr   = shap.maskers.Independent(X_train_scaled, max_samples=500)
explainer_lr    = shap.LinearExplainer(best_lr, background_lr)
sv_lr = explainer_lr.shap_values(X_test_scaled)

# --- 2d. SVM — KernelExplainer ---
print("Computing SVM SHAP values (approx. 2-3 mins)...")
background_svm  = shap.kmeans(X_train_scaled, 50)
explainer_svm   = shap.KernelExplainer(best_svm.predict_proba, background_svm)
np.random.seed(SEED)
idx_sample      = np.random.choice(len(X_test_scaled), size=200, replace=False)
X_test_sample   = X_test_scaled[idx_sample]
shap_values_svm = explainer_svm.shap_values(X_test_sample)

# Fix: Handle all possible SHAP output formats for SVM
if isinstance(shap_values_svm, list):
    # Old SHAP format: list of arrays
    sv_svm_sample = shap_values_svm[1]
elif len(shap_values_svm.shape) == 3:
    # New SHAP format: (samples, features, classes)
    sv_svm_sample = shap_values_svm[:, :, 1]
else:
    # Already 2D (samples, features)
    sv_svm_sample = shap_values_svm

print("SVM SHAP done.")

# Verification of shapes
print(f"RF shape:  {sv_rf.shape}")
print(f"XGB shape: {sv_xgb.shape}")
print(f"LR shape:  {sv_lr.shape}")
print(f"SVM shape: {sv_svm_sample.shape}")

# ---- Importance Calculation and Plotting ----
def mean_abs_shap(sv, feature_names):
    # Ensure we are taking the mean across samples to get a 1D vector of length 31
    return pd.Series(np.abs(sv).mean(axis=0), index=feature_names)

shap_importance_rf  = mean_abs_shap(sv_rf,  feature_names)
shap_importance_xgb = mean_abs_shap(sv_xgb, feature_names)
shap_importance_lr  = mean_abs_shap(sv_lr,  feature_names)
shap_importance_svm = mean_abs_shap(sv_svm_sample, feature_names)

# Build comparison and plot
top15 = shap_importance_rf.nlargest(15).index.tolist()
shap_comparison = pd.DataFrame({
    'Random Forest': shap_importance_rf[top15],
    'XGBoost':       shap_importance_xgb[top15],
    'Logistic Reg':  shap_importance_lr[top15],
    'SVM':           shap_importance_svm[top15]
}, index=top15)

fig, ax = plt.subplots(figsize=(10, 7))
shap_comparison.plot(kind='barh', ax=ax, width=0.75)
ax.invert_yaxis()
ax.set_title('Top 15 Features — Mean |SHAP| Comparison')
plt.tight_layout()
plt.show()

In [ ]:
import itertools
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

# Generate missing predictions for balanced models
pred_lr_bal = best_lr_bal.predict(X_test_scaled)
pred_rf_bal = best_rf_bal.predict(X_test_scaled)
pred_svm_bal = best_svm_bal.predict(X_test_scaled)

# Create a dictionary of all predictions
all_predictions = {
    'LR_Unbalanced': pred_lr,
    'RF_Unbalanced': pred_rf,
    'SVM_Unbalanced': pred_svm,
    'LR_Balanced': pred_lr_bal,
    'RF_Balanced': pred_rf_bal,
    'SVM_Balanced': pred_svm_bal
}

# Iterate through all unique pairs of classifiers
for (name_a, preds_a), (name_b, preds_b) in itertools.combinations(all_predictions.items(), 2):
    print(f"\n--- Comparing {name_a} and {name_b} ---")

    # Create boolean arrays for correctness
    correct_a = (preds_a == y_test.values)
    correct_b = (preds_b == y_test.values)

    # Build the 4 cells of the contingency table
    n_00 = ((correct_a == True) & (correct_b == True)).sum()
    n_01 = ((correct_a == True) & (correct_b == False)).sum()
    n_10 = ((correct_a == False) & (correct_b == True)).sum()
    n_11 = ((correct_a == False) & (correct_b == False)).sum()

    # Create the contingency table
    contingency_table = [[n_00, n_01],
                         [n_10, n_11]]

    print(f"Contingency Table:\n{np.array(contingency_table)}")

    # Perform McNemar's test
    result = mcnemar(contingency_table, exact=False)
    print(f"McNemar's Test Result:\nStatistic: {result.statistic:.4f}, p-value: {result.pvalue:.4f}")

    alpha = 0.05
    if result.pvalue < alpha:
        print("Conclusion: Statistically significant difference between the two classifiers (reject H0).")
    else:
        print("Conclusion: No statistically significant difference between the two classifiers (fail to reject H0).")

In [ ]:
print(results_df['Model'].unique())

In [ ]:
# Define the cohens_d function
def cohens_d(scores_a, scores_b):
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)

    std_a = np.std(scores_a, ddof=1)
    std_b = np.std(scores_b, ddof=1)

    # Handle cases where std_a or std_b might be zero or extremely small
    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)

    if pooled_std < 1e-9: # A very small number close to zero
        # If pooled_std is effectively zero, and means are different, d is very large/infinite.
        # If means are also identical, d is zero.
        if np.isclose(mean_a, mean_b):
            return 0.0
        else:
            # Return a very large number to indicate extreme effect if std is zero but means differ
            return np.inf if (mean_a - mean_b) > 0 else -np.inf

    d = (mean_a - mean_b) / pooled_std
    return d

# Perform 10-fold cross-validation on the training set for each classifier
# Use X_train_scaled and y_train as best_lr, best_rf, best_svm were found using this data.
scores_lr = cross_val_score(best_lr, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_rf = cross_val_score(best_rf, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_svm = cross_val_score(best_svm, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)

print("10-fold cross-validation F1-macro scores:")
print(f"Logistic Regression (Unbalanced): {scores_lr}")
print(f"Random Forest (Unbalanced): {scores_rf}")
print(f"SVM (Unbalanced): {scores_svm}")

# Compute Cohen's d for each pair
d_lr_rf = cohens_d(scores_lr, scores_rf)
d_lr_svm = cohens_d(scores_lr, scores_svm)
d_rf_svm = cohens_d(scores_rf, scores_svm)

print("\n--- Cohen's d Values and Interpretation ---")
print("\nCohen's d: A measure of effect size. General guidelines:")
print("- Small effect: d = 0.2")
print("- Medium effect: d = 0.5")
print("- Large effect: d = 0.8")

# Interpretation function
def interpret_cohens_d(d_value, name_a, name_b):
    print(f"\n{name_a} vs {name_b}:")
    print(f"  Cohen's d: {d_value:.4f}")
    if np.isinf(d_value):
        print("  Interpretation: Infinite effect size due to zero variance in scores. This indicates a severe issue with one of the score sets (likely constant scores).")
    elif abs(d_value) >= 0.8:
        print("  Interpretation: Large effect size.")
    elif abs(d_value) >= 0.5:
        print("  Interpretation: Medium effect size.")
    elif abs(d_value) >= 0.2:
        print("  Interpretation: Small effect size.")
    else:
        print("  Interpretation: Negligible effect size.")

interpret_cohens_d(d_lr_rf, "Logistic Regression (Unbalanced)", "Random Forest (Unbalanced)")
interpret_cohens_d(d_lr_svm, "Logistic Regression (Unbalanced)", "SVM (Unbalanced)")
interpret_cohens_d(d_rf_svm, "Random Forest (Unbalanced)", "SVM (Unbalanced)")

print("\n--- Important Note on SVM Scores ---")
print("The cross-validation F1-macro scores for SVM (Unbalanced) are very low and nearly constant across folds.")
print("This results in an extremely small standard deviation for SVM scores, leading to exceptionally large Cohen's d values")
print("when compared against other classifiers, and potentially infinite if the std was exactly zero. This suggests an issue")
print("with the cross-validation setup or the SVM model's performance on the training folds, making the Cohen's d")
print("interpretation for pairs involving SVM unreliable in terms of magnitude.")
print("It is unusual for best_svm, which achieved a high F1-macro score in GridSearchCV, to perform so poorly and consistently low")
print("during cross_val_score, suggesting a potential discrepancy in how the metrics or data splits are handled.")

In [ ]:
import itertools
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import cross_val_score
import numpy as np

# Define the cohens_d function
def cohens_d(scores_a, scores_b):
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)

    std_a = np.std(scores_a, ddof=1)
    std_b = np.std(scores_b, ddof=1)

    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)

    if pooled_std < 1e-9: # A very small number close to zero
        if np.isclose(mean_a, mean_b):
            return 0.0
        else:
            return np.inf if (mean_a - mean_b) > 0 else -np.inf

    d = (mean_a - mean_b) / pooled_std
    return d

# Perform 10-fold cross-validation on the training set for each classifier
# Use X_train_scaled and y_train for unbalanced models
scores_lr_unbal = cross_val_score(best_lr, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_rf_unbal = cross_val_score(best_rf, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_svm_unbal = cross_val_score(best_svm, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)

# Use X_train_balanced and y_train_balanced for balanced models
scores_lr_bal = cross_val_score(best_lr_bal, X_train_balanced, y_train_balanced, scoring='f1_macro', cv=10, n_jobs=-1)
scores_rf_bal = cross_val_score(best_rf_bal, X_train_balanced, y_train_balanced, scoring='f1_macro', cv=10, n_jobs=-1)
scores_svm_bal = cross_val_score(best_svm_bal, X_train_balanced, y_train_balanced, scoring='f1_macro', cv=10, n_jobs=-1)

all_scores = {
    'LR_Unbalanced': scores_lr_unbal,
    'RF_Unbalanced': scores_rf_unbal,
    'SVM_Unbalanced': scores_svm_unbal,
    'LR_Balanced': scores_lr_bal,
    'RF_Balanced': scores_rf_bal,
    'SVM_Balanced': scores_svm_bal
}

# Prepare predictions for McNemar's test (predictions on X_test_scaled)
all_predictions = {
    'LR_Unbalanced': best_lr.predict(X_test_scaled),
    'RF_Unbalanced': best_rf.predict(X_test_scaled),
    'SVM_Unbalanced': best_svm.predict(X_test_scaled),
    'LR_Balanced': best_lr_bal.predict(X_test_scaled),
    'RF_Balanced': best_rf_bal.predict(X_test_scaled),
    'SVM_Balanced': best_svm_bal.predict(X_test_scaled)
}

combined_results = []
alpha = 0.05

# Iterate through all unique pairs of classifiers
for (name_a, scores_a), (name_b, scores_b) in itertools.combinations(all_scores.items(), 2):
    # Cohen's d calculation
    d_value = cohens_d(scores_a, scores_b)
    cohens_d_interpretation = ""
    if np.isinf(d_value):
        cohens_d_interpretation = "Infinite effect size (likely due to zero variance in scores)."
    elif abs(d_value) >= 0.8:
        cohens_d_interpretation = "Large effect size."
    elif abs(d_value) >= 0.5:
        cohens_d_interpretation = "Medium effect size."
    elif abs(d_value) >= 0.2:
        cohens_d_interpretation = "Small effect size."
    else:
        cohens_d_interpretation = "Negligible effect size."

    # McNemar's test calculation
    preds_a = all_predictions[name_a]
    preds_b = all_predictions[name_b]

    correct_a = (preds_a == y_test)
    correct_b = (preds_b == y_test)

    n_00 = ((correct_a == True) & (correct_b == True)).sum()
    n_01 = ((correct_a == True) & (correct_b == False)).sum()
    n_10 = ((correct_a == False) & (correct_b == True)).sum()
    n_11 = ((correct_a == False) & (correct_b == False)).sum()

    contingency_table = [[n_00, n_01],
                         [n_10, n_11]]

    # Handle cases where mcnemar might fail with very small N
    try:
        mcnemar_result = mcnemar(contingency_table, exact=False)
        mcnemar_statistic = mcnemar_result.statistic
        mcnemar_pvalue = mcnemar_result.pvalue
        mcnemar_conclusion = "Statistically significant difference (reject H0)." if mcnemar_pvalue < alpha else "No statistically significant difference (fail to reject H0)."
    except ValueError:
        mcnemar_statistic = 'N/A'
        mcnemar_pvalue = 'N/A'
        mcnemar_conclusion = "McNemar's test could not be performed due to insufficient data or conditions."

    combined_results.append({
        'Pair': f"{name_a} vs {name_b}",
        'McNemar_Statistic': mcnemar_statistic,
        'McNemar_PValue': mcnemar_pvalue,
        'McNemar_Conclusion': mcnemar_conclusion,
        'Cohens_d': d_value,
        'Cohens_d_Interpretation': cohens_d_interpretation
    })

# Generate Markdown report
markdown_report = "# Combined Statistical Analysis: McNemar's Test and Cohen's d\n\n"
markdown_report += "This section presents a combined analysis of classifier performance differences using McNemar's Test for statistical significance and Cohen's d for effect size. The F1-macro scores from 10-fold cross-validation on the training set were used for Cohen's d, while predictions on the test set were used for McNemar's Test.\n\n"

# Add a note about the SVM cross-validation scores, if still relevant based on the CV scores.
if np.std(scores_svm_unbal) < 0.01 or np.std(scores_svm_bal) < 0.01: # Small std indicates potential issue
    markdown_report += "**Important Note on SVM Cross-Validation Scores:** It was observed that the cross-validation F1-macro scores for SVM models (both unbalanced and balanced) are very consistent, resulting in extremely small standard deviations. This leads to exceptionally large Cohen's d values when compared against other classifiers, and potentially infinite if the standard deviation was exactly zero. This suggests a potential issue with the cross-validation setup or the SVM model's performance stability across training folds, making the Cohen's d interpretation for pairs involving SVM unreliable in terms of magnitude. The high GridSearchCV scores for SVM suggest good performance, but the CV scores variability is low.\n\n"

for result in combined_results:
    markdown_report += f"## Pair: {result['Pair']}\n"
    markdown_report += f"### McNemar's Test\n"
    markdown_report += f"- **Statistic (χ²):** {result['McNemar_Statistic']:.4f}\n"
    markdown_report += f"- **p-value:** {result['McNemar_PValue']:.4f}\n"
    markdown_report += f"- **Conclusion (α={alpha}):** {result['McNemar_Conclusion']}\n"
    markdown_report += f"### Cohen's d\n"
    markdown_report += f"- **Effect Size (d):** {result['Cohens_d']:.4f}\n"
    markdown_report += f"- **Interpretation:** {result['Cohens_d_Interpretation']}\n\n"

# Print the markdown report (this will be rendered by the notebook)
print(markdown_report)


In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
ax = plt.gca()

# Plot ROC curve for Logistic Regression
lr_disp = RocCurveDisplay.from_estimator(best_lr, X_test_scaled, y_test, name='Logistic Regression', alpha=0.8, lw=2, ax=ax, ls='-')

# Plot ROC curve for Random Forest
rf_disp = RocCurveDisplay.from_estimator(best_rf, X_test_scaled, y_test, name='Random Forest', alpha=0.8, lw=2, ax=ax, ls='--')

# Plot ROC curve for SVM
svm_disp = RocCurveDisplay.from_estimator(best_svm, X_test_scaled, y_test, name='SVM', alpha=0.8, lw=2, ax=ax, ls='-.')

# Add the diagonal dashed black line for a random classifier
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='black', label='Random Classifier (AUC = 0.5)', alpha=0.8)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Classification Models')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('fig2_roc_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Define class labels for better readability
class_names = ['Legitimate', 'Phishing'] # Assuming -1 maps to Legitimate, 1 maps to Phishing

# Create a figure with 1 row and 3 columns for the subplots
fig, axes = plt.subplots(1, 3, figsize=(21, 6)) # Increased width for 3 subplots

# List of models and their predictions
models = {
    'Logistic Regression': pred_lr,
    'Random Forest': pred_rf,
    'SVM': pred_svm
}

for i, (model_name, predictions) in enumerate(models.items()):
    cm = confusion_matrix(y_test, predictions)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=class_names, yticklabels=class_names)

    axes[i].set_title(f'{model_name} Confusion Matrix')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('fig3_confusion_matrices.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances from the best Random Forest model
feature_importances = best_rf.feature_importances_

# Create a Series linking feature names to their importances
features_df = pd.Series(feature_importances, index=X.columns)

# Sort in descending order and take the top 15
top_15_features = features_df.nlargest(15)

# Create the horizontal bar chart
plt.figure(figsize=(10, 7)) # Adjust size for better readability of 15 features
sns.barplot(x=top_15_features.values, y=top_15_features.index, palette='viridis')

# Invert y-axis to have the most important feature at the top
plt.gca().invert_yaxis()

plt.xlabel('Feature Importance (Gini Impurity Decrease)')
plt.ylabel('Feature Name')
plt.title('Top 15 Feature Importances from Random Forest Classifier')
plt.tight_layout()
plt.savefig('fig4_feature_importance.pdf', dpi=300, bbox_inches='tight')
plt.show()

caption = "Figure 4: Top 15 feature importances as determined by the Random Forest classifier, based on Gini impurity decrease. Features are sorted in descending order of importance, with the most impactful feature at the top. This visualization helps identify the most significant characteristics distinguishing phishing websites from legitimate ones. The dataset size is n=11055."
print(caption)